# RAG with OpenAI embeddings + chat model

Builds a FAISS store from `data/1. dietary supplements - for whom.pdf` with OpenAI embeddings and answers questions with an OpenAI chat model.

**Cost warning:** the cells that build the index and ask questions call the paid OpenAI API.

**Setup**
1. `pip install -r requirements-openai.txt` (Python 3.12 virtual environment).
2. Copy `.env.example` to `.env` and set `OPENAI_API_KEY` and `OPENAI_CHAT_MODEL` (a model currently available to your account). `.env` is git-ignored; never paste a key into a notebook.
3. Restart the kernel after editing `.env`.

*Author: Ratnesh Pathak.*  *Not executed during the repository audit* (no API key was used), so outputs are cleared.

**What changed from the original:** the old notebook read a non-standard `OPENAI_KEY`, imported classes from deprecated `langchain.*` paths, used the deprecated `RetrievalQA` chain, split text into 200-character fragments with no source metadata (so answers could not be cited), and hard-coded `gpt-3.5-turbo-0125`, which OpenAI lists for shutdown on 23 Oct 2026.

### Optional: what an embedding looks like
Needs `pip install sentence-transformers` (large download; not part of the requirements). Safe to skip.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    print(model.encode(["This is an example sentence"])[0][:8], "...")
except ImportError:
    print("sentence-transformers is not installed - skipping this optional demo.")

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable whether the notebook is started from the repo root or a sub-folder.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "rag_chatbot").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from rag_chatbot import config
from rag_chatbot.documents import load_documents, split_documents
from rag_chatbot.indexing import build_index, load_index
from rag_chatbot.qa import answer_question

config.load_env()
config.get_openai_api_key()          # raises a clear error if missing; the key is never printed
EMBED_MODEL = config.get_openai_embedding_model()
INDEX_DIR = config.VECTORSTORES_DIR / "supplements_openai"
print("Embedding model:", EMBED_MODEL)

### Load and split the PDF (now with `source` and `page` metadata)

In [ ]:
SOURCE = config.DEFAULT_PDF
chunks = split_documents(load_documents([SOURCE]), chunk_size=1000, chunk_overlap=150)
print(len(chunks), "chunks;", chunks[0].metadata)

### Embed and save (paid API calls)

In [ ]:
embeddings = config.make_openai_embeddings(EMBED_MODEL)
manifest = build_index(chunks, embeddings, INDEX_DIR, provider="openai", embedding_model=EMBED_MODEL,
                       source_files=[SOURCE], chunk_size=1000, chunk_overlap=150)
manifest

### Ask questions (paid API calls)

In [ ]:
store = load_index(INDEX_DIR, embeddings, expected_embedding_model=EMBED_MODEL)
llm = config.make_openai_llm()       # model name comes from OPENAI_CHAT_MODEL

for q in ["What does the review say about vitamin D supplementation?",
          "What is Neapolitan pizza?"]:     # not in the document -> expect a refusal
    ans = answer_question(q, store, llm, k=4)
    print("Q:", q); print(ans.text); print("Sources:", ans.sources, "\n")

**Troubleshooting:** `OPENAI_API_KEY is not set` -> create `.env` and restart the kernel. `401` -> key invalid/revoked. `404 model not found` -> change `OPENAI_CHAT_MODEL`. `Index was built with embedding model ...` -> you changed `OPENAI_EMBEDDING_MODEL` after building; rebuild.